# Experiment 2: tiling and detection on a real Polish statute (KPC)

The same experiment as notebook 1, now on real, messy legal text: the first 50 pages of the Polish Code of Civil Procedure (*Kodeks postępowania cywilnego*, Dz. U. 2023 poz. 1550). Three things to watch:

1. **Junk rejection.** Real documents carry page furniture and front matter. We check what the extractor throws away and what it keeps.
2. **Tiling.** How the statute splits into semantic units and Louvain communities.
3. **Detection in Polish.** A proposed change to a requirement, run end to end through `detect_changes` with `POLISH_PROMPTS` and a local LLM. The document is Polish, so the prompts are Polish; the library already ships them, we just pass them in. We drive the passes with a local instruct model (`qwen3:30b-a3b-instruct`) rather than the default `gpt-oss:20b`: the reasoning-style default occasionally returns an empty structured answer through Ollama, which breaks the strict schema parsing, and an instruct model is steadier at structured output.

A note on honesty up front: this particular PDF has **no table of contents** anywhere, so there is no TOC to reject. What it does carry, and what we show being filtered, is the running page header repeated on every page, the front-matter announcement (*obwieszczenie*) with its long list of amending laws, and the repealed-article stubs (*uchylony*).

In [ ]:
from collections import Counter
from pathlib import Path

import fitz
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch

from text_change_detector import Change, detect_changes, POLISH_PROMPTS, tile
from text_change_detector.shared.embedder import SentenceTransformerEmbedder
from text_change_detector.shared.graph import knn_sparsify
from text_change_detector.tiling.extraction.pdf import extract_pdf, read_blocks, running_furniture
from text_change_detector.tiling.extraction.shared import load_nlp

In [ ]:
def pick_device():
    if torch.cuda.is_available() and torch.cuda.mem_get_info()[0] > 9 * 1024 ** 3:
        return 'cuda'

    return 'cpu'


DEVICE = pick_device()

print('device:', DEVICE)

## Load the first 50 pages

In [ ]:
PDF = next(p for p in [Path('data/DU_2023_1550_KPC.pdf'),
                       Path('experiments/data/DU_2023_1550_KPC.pdf')] if p.exists())
PAGES = 50
src = fitz.open(PDF)
fragment = fitz.open()

fragment.insert_pdf(src, from_page=0, to_page=PAGES - 1)

print('source pages:', src.page_count, '| fragment pages:', fragment.page_count)

## 1. What the extractor throws away

`read_blocks` reads every text block; `running_furniture` finds the blocks that repeat across at least half the pages (headers, footers, page numbers) and marks them for removal; `extract_pdf` then keeps only sentence-like content, turning headings into section paths and label fragments into payload. Below: how many raw blocks came in, how many segments survived, and exactly which furniture was dropped.

In [ ]:
blocks = read_blocks(fragment)
furniture = running_furniture(blocks, fragment.page_count)
nlp = load_nlp('pl_core_news_sm')
segments = extract_pdf(fragment, nlp)

print(f'raw text blocks read : {len(blocks)}')
print(f'segments kept        : {len(segments)}')
print(f'dropped (furniture + non-content + folded into sections/payload): {len(blocks) - len(segments)}')
print(f'\nrunning furniture removed ({len(furniture)}):')

for f in furniture:
    print('   ', repr(f))

leaks = [s for s in segments if 'Dziennik Ustaw' in s.text or 'Poz. 1550' in s.text]

print(f'\nresidual header fragments that still slipped through: {len(leaks)}')

for s in leaks:
    print('   ', repr(s.text[:100]))

### Front matter and repealed stubs

The running header is gone, but the *obwieszczenie* front matter is genuine prose (it has finite verbs), so it is kept rather than dropped. What happens to it is more interesting than deletion: it lands under its own section label, distinct from the code articles, so later it clusters on its own instead of contaminating the statute's units. The repealed-article stubs (*(uchylony)*), which carry no finite verb, never become standalone content.

In [ ]:
sec_counts = Counter(s.section for s in segments)

print(f'distinct section paths: {len(sec_counts)}')

for sec, n in list(sec_counts.items())[:16]:
    print(f'  {n:3d}  {sec[:74]}')

uchylony_here = [s for s in segments if s.text.strip().endswith('(uchylony)')]

print(f"\nstandalone '(uchylony)' content segments kept: {len(uchylony_here)}")

## 2. Tiling the statute

We tile the segments we just inspected (passing the list straight in, so extraction is not repeated), with the production embedder.

In [ ]:
embedder = SentenceTransformerEmbedder(device=DEVICE, batch_size=8)
tiling = tile(segments, embedder=embedder)
n_units = sum(len(c.units) for c in tiling.communities)
sizes = sorted((len(c.units) for c in tiling.communities), reverse=True)
lengths = [len(u.sentences) for c in tiling.communities for u in c.units]

print(f'semantic units: {n_units}')
print(f'communities   : {len(tiling.communities)}')
print(f'unit length (sentences): min {min(lengths)}, median {int(np.median(lengths))}, max {max(lengths)}')
print(f'largest communities (unit counts): {sizes[:10]}')

A couple of sample units, to eyeball whether they are coherent:

In [ ]:
sample = [u for c in tiling.communities for u in c.units]

for u in sample[:3]:
    print(f'--- unit {u.id}  [{u.section[:60]}] ---')

    for s in u.sentences:
        print('   ', s[:110])

    print()

### Visualising the units

Unit length distribution, community sizes, and the kNN relation graph coloured by community.

In [ ]:
units = sorted((u for c in tiling.communities for u in c.units), key=lambda u: u.id)
unit_texts = [' '.join([*u.sentences, *u.payload]) for u in units]
emb = embedder.encode(unit_texts, normalize_embeddings=True)
sim = emb @ emb.T
adj = knn_sparsify(sim, 5)
G = nx.from_numpy_array(adj)
comm_of = {u.id: c.id for c in tiling.communities for u in c.units}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(lengths, bins=range(1, max(lengths) + 2), color='#4a90a4', align='left')
axes[0].set_title('unit length (sentences)')
axes[0].set_xlabel('sentences per unit')
axes[0].set_ylabel('units')
axes[1].bar(range(len(sizes)), sizes, color='#4a90a4')
axes[1].set_title('community sizes')
axes[1].set_xlabel('community (rank)')
axes[1].set_ylabel('units')

pos = nx.spring_layout(G, seed=0)

nx.draw(G, pos, ax=axes[2], node_color=[comm_of[n] for n in G.nodes], cmap='tab20',
        node_size=40, width=0.3, edge_color='#dddddd', with_labels=False)
axes[2].set_title('kNN relation graph, coloured by community')
plt.show()

### Did the front matter cluster on its own?

For each community we count how many of its units come from the *obwieszczenie* front matter versus the code body. If the front matter keeps to its own community, it did not leak into the articles.

In [ ]:
def is_frontmatter(section):
    return section.startswith('Poz. 1550') or 'ogłoszenia jednolitego' in section

for c in tiling.communities:
    fm = sum(is_frontmatter(u.section) for u in c.units)
    code_units = len(c.units) - fm
    tag = 'FRONT MATTER' if fm and not code_units else ('mixed' if fm else 'code')

    print(f'community {c.id:2d}: {len(c.units):3d} units  | front-matter {fm:3d}  code {code_units:3d}  [{tag}]')

### A caveat before detection: section labels are noisy on this PDF

About half the units carry a section path that ends in a repealed-article heading such as `Art. 121. (uchylony)`. On this document the heading heuristics mistake a bold repealed-article line for a live section header, and it then sticks to the articles that follow. The unit text is correct; only the `section` label is unreliable here. This is a real limit of heuristic heading detection on a statute where article numbers sit inline with their content. Keep it in mind when reading the sections printed next to the detection hits below.

In [ ]:
stale = [u for c in tiling.communities for u in c.units if '(uchylony)' in u.section]

print(f'units carrying a repealed-heading section label: {len(stale)} / {n_units}')

## 3. Detection in Polish

We now propose a change to a requirement and run it end to end. The change (below) says a statement of claim (*pozew*) must additionally carry the plaintiff's email address, on pain of being returned as formally deficient. `detect_changes` rebuilds the relation graph, ranks the units the change resembles plus their graph neighbours, then has the local LLM rate each candidate, verify the strong ones with a skeptical pass, and draft a minimal merged text.

We free the embedder first, so the embedder and the LLM are not on the GPU at the same time (the library does this for us when it owns the embedder, which is the path we take here by not passing one to `detect_changes`).

In [ ]:
embedder.close()

change = Change(
    name='pozew-adres-email',
    text=(
        'Pozew musi dodatkowo zawierać adres poczty elektronicznej powoda, na który sąd '
        'doręcza pisma w toku postępowania. Brak wskazania adresu poczty elektronicznej '
        'stanowi brak formalny pozwu skutkujący jego zwrotem.'
    ),
)
result = detect_changes(tiling, [change], prompts=POLISH_PROMPTS)
impact = result.changes[0]

print('primary units (direct hits):', impact.primary)
print('ripple units (one hop)     :', impact.ripple)
print('candidates reviewed by LLM :', len(impact.relations))

### How the LLM rated each reviewed unit

In [ ]:
for r in impact.relations:
    mark = {True: 'CONFIRMED', False: 'rejected', None: '-'}[r.verified]

    print(f'[{r.relation:6s}] unit {r.unit_id:3d}  verify={mark}')
    print(f'    topic: {r.unit_topic}')
    print(f'    why  : {r.justification[:200]}')

    if r.verify_reason:
        print(f'    check: {r.verify_reason[:200]}')

    print()

### The confirmed edits (`current_text` -> `merged_text`)

In [ ]:
if not impact.suggestions:
    print('no verified-strong suggestions')

for s in impact.suggestions:
    print(f'=== unit {s.unit_id}  [{s.section[:60]}] ===')
    print('  ADDED :', s.added[:200])
    print('  BEFORE:', s.current_text[:400])
    print('  AFTER :', s.merged_text[:400])
    print()

## Observations

**Tiling.** The first 50 pages produced **347 semantic units in 11 communities.** Sampled units read coherently: each is a self-contained run of one article's paragraphs.

**Junk rejection.**
- The running header `Dziennik Ustaw - N - Poz. 1550`, repeated on every page, is detected as furniture and removed; only a couple of variant fragments slip through.
- There is no table of contents in this document, so there is none to reject.
- The front-matter announcement (*obwieszczenie*) is genuine prose, so it is kept rather than deleted, but it lands under its own `Poz. 1550 > ...` section and clusters largely on its own, apart from the code articles.
- Repealed-article stubs that carry no finite verb never become standalone content units.

**An honest extraction limitation.** About half the units (178 of 347 in this run) carry a section label ending in a repealed-article heading. On this PDF the heading heuristics mistake a bold repealed-article line for a live section header, and it sticks to the articles that follow. The unit text is correct; only the `section` label is unreliable here.

**Detection.** The change (a new email requirement for the *pozew*, on pain of return) was matched to the right neighbourhood: the confirmed units are about the contents of a statement of claim, the return of procedurally deficient filings, and electronic service. Unit 344, correctly sectioned `DZIAŁ II > Pozew`, is the clean bullseye. But with the general-purpose prompts the pass is over-inclusive: 15 of the 18 reviewed candidates were confirmed, including tangential units about filing through the teleinformatic system and about court fees. That is exactly the effect the README note warns about: the mechanism is domain-agnostic, but the shipped prompts are general, so in a specialised domain like law they over-match. Tightening the prompts (for instance, telling the model to confirm only the article that states the pozew's required contents, and to disregard repealed provisions) is how you would sharpen this.